# 6) The learning rate: SGD and Adam

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import MNIST

import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
class MNISTDataset(Dataset):                                    # dataset constructor

    def __init__(self):

        data = MNIST('mnist', download=True)
        self.images, self.labels = data.data, data.targets      # return dataset X, y


    def __len__(self):

        return len(self.images)                                 # number of features

    
    def __getitem__(self, idx: int):                            # return element indexed idx in X, y

        image = self.images[idx].float().view(28*28) / 255.0
        label = self.labels[idx]

        return image, label
        

In [4]:

dataset = MNISTDataset()       # create dataset


In [5]:
#select indexes to split dataset for model selection

idx_train, idx_valid = train_test_split(range(len(dataset)), test_size=0.1, random_state=0)

train_dataset = Subset(dataset, idx_train)        
valid_dataset = Subset(dataset, idx_valid)

In [6]:
def collate_fn(batch):                      # returns X, y in batch

    images = []
    labels = []

    for image, label in batch:
        images.append(image)
        labels.append(label)

    images = torch.stack(images, dim=0)     # size : (32, 784)
    labels = torch.tensor(labels)           # size : (32,)

    return images, labels


In [7]:
# CREATE DataLoader dataset

train_dataloader = DataLoader(
    train_dataset, 
    batch_size=32,
    shuffle=True,
    num_workers=0,
    drop_last=True,
    collate_fn=collate_fn
)

In [8]:
# DATALOADER VALIDATION DATASET

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
    drop_last=False
)

In [9]:
# NN CONSTRUCTOR

class TorchMLP(torch.nn.Module):

    def __init__(self, input_dim, hidden_dims, num_classes):
        super().__init__()
        
        all_dims = [input_dim] + list(hidden_dims) + [num_classes]
        
        layers = []
        for dim1, dim2 in zip(all_dims[:-1], all_dims[1:]):
            layers.append(torch.nn.Linear(dim1, dim2))
            layers.append(torch.nn.ReLU())
            
        layers = layers[:-1]                               # take off ReLu at last

        self.model = torch.nn.Sequential(*layers)          # model where layers are called sequentially
        
    def forward(self, x):
        return self.model(x)


In [10]:
# CALL MODEL, LOSS, OPTIMIZER ADAM

modele = TorchMLP(784, [100, 200, 100], 10)#.cuda()

loss_function = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(modele.parameters(), lr=0.002)


In [11]:
# TRAINING AND VALIDATION OF NN

for epoch in range(10):

    # TRAINING
    for images, labels in train_dataloader:

        optimizer.zero_grad()
        y_pred = modele(images)
        loss_result = loss_function(y_pred, labels)
        loss_result.backward()
        optimizer.step()

    # EVALUATION 
    all_preds = []
    all_labels = []

    with torch.no_grad():                                  # cancel gradient, to save memory

        for images, labels in valid_dataloader:            # in validation set, compute accuracy

            logits = modele(images)
            all_preds.extend(torch.argmax(logits, dim=1))
            all_labels.extend(labels)

    print(accuracy_score(all_labels, all_preds))
 

0.9596666666666667
0.9721666666666666
0.9733333333333334
0.9726666666666667
0.9761666666666666
0.9763333333333334
0.972
0.978
0.9775
0.9781666666666666


In [12]:
for epoch in range(10):
    # Tout le tour du jeu d'entraînement
    for images, labels in train_dataloader:
        optimizer.zero_grad()
        y_pred = modele(images)#.to('cuda'))
        one_hot_labels = torch.zeros((len(labels), 10), dtype=torch.float32)                # ONE_HOT_LABELS
        one_hot_labels[torch.arange(len(labels)), labels] = 1
        loss_result = loss_function(y_pred, one_hot_labels)#.to('cuda'))
        loss_result.backward()
        optimizer.step()
    # On fait la prévision sur le jeu de validation
    # et on sauvegarde les résultats
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in valid_dataloader:
            logits = modele(images)#.to('cuda'))
            all_preds.extend(torch.argmax(logits, dim=1).to('cpu'))
            all_labels.extend(labels)
    print(accuracy_score(all_labels, all_preds))

0.9745
0.978
0.9798333333333333
0.9781666666666666
0.9785
0.9791666666666666
0.9798333333333333
0.9791666666666666
0.975
0.9763333333333334
